# 1. Ingeniería de Características y Cálculo de Lead Time para Procesos de Adquisición

**Proyecto:** Predicción Temprana del Riesgo de Sobrecosto en Contratos de Medicamentos — EsSalud
**Autor:** Estudiante de Ingeniería de Software — Universidad Nacional Mayor de San Marcos (UNMSM)
**Módulo:** Preprocesamiento avanzado, ingeniería de variables temporales (Lead Time) y codificación categórica

---

## Índice de Contenido

1. [Introducción y Justificación Operacional](#1.-Introducción-y-Justificación-Operacional)
2. [Carga del Dataset y Análisis Exploratorio de Fechas](#2.-Carga-del-Dataset-y-Análisis-Exploratorio-de-Fechas)
3. [Conversión Segura de Tipos Temporales](#3.-Conversión-Segura-de-Tipos-Temporales)
4. [Cálculo e Ingeniería de las Variables de Lead Time](#4.-Cálculo-e-Ingeniería-de-las-Variables-de-Lead-Time)
5. [Tratamiento de Inconsistencias Lógicas (Fechas Invertidas)](#5.-Tratamiento-de-Inconsistencias-Lógicas-(Fechas-Invertidas))
6. [Tratamiento de Outliers mediante IQR](#6.-Tratamiento-de-Outliers-mediante-IQR)
7. [Codificación de Variables Categóricas](#7.-Codificación-de-Variables-Categóricas)
8. [Escalamiento de Variables Numéricas](#8.-Escalamiento-de-Variables-Numéricas)
9. [Exportación de la Matriz Final](#9.-Exportación-de-la-Matriz-Final)


## 1. Introducción y Justificación Operacional

En los procesos de adquisición pública de medicamentos gestionados bajo el régimen de contrataciones del Estado peruano (EsSalud), el ciclo de vida de un contrato atraviesa tres hitos administrativos críticos: la **convocatoria**, el otorgamiento de la **buena pro** y la **suscripción** del contrato. La literatura en gestión de riesgos de adquisiciones (*Procurement Risk Management*) establece que la dilatación temporal entre estos hitos —el denominado *Lead Time*— es un proxy estructural de fricciones administrativas, negociaciones prolongadas, impugnaciones o restricciones presupuestales que incrementan la probabilidad de renegociación contractual (adendas) y, por extensión, el riesgo de sobrecosto.

Formalmente, se definen tres tiempos de ciclo:

$$LT_{adjudicación} = FechaBuenaPro - FechaConvocatoria$$
$$LT_{formalización} = FechaSuscripción - FechaBuenaPro$$
$$LT_{total} = FechaSuscripción - FechaConvocatoria = LT_{adjudicación} + LT_{formalización}$$

La hipótesis central de ingeniería de características de este módulo es que un cuello de botella pronunciado en la fase de **formalización** ($LT_{formalización}$) es un predictor de mayor peso relativo que la fase de adjudicación, dado que refleja fricciones en la negociación de cláusulas contractuales, garantías y condiciones de entrega —terreno fértil para adendas posteriores por ajuste de precios o plazos.


## 2. Carga del Dataset y Análisis Exploratorio de Fechas

Se carga el dataset consolidado `staging_flat.csv`, verificando la integridad estructural (dimensionalidad, tipos de dato crudos, valores nulos) antes de cualquier transformación.


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

RAW_PATH = '../data/staging_flat.csv'
df = pd.read_csv(RAW_PATH)

print(f"Dimensiones del dataset: {df.shape[0]} registros x {df.shape[1]} columnas")
df.head()

Dimensiones del dataset: 12500 registros x 17 columnas


,fecha_convocatoria,fecha_buena_pro,fecha_suscripcion,lead_time_adjudicacion,lead_time_formalizacion,lead_time_total_proceso,red_asistencial,dpto_entrega_item,tipo_proveedor,denominacion_dci,tipo_proceso_seleccion,es_consorcio,es_uso_critico,es_uso_intrahospitalario,monto_adjudicado_soles,anio,flag_tiene_adenda
0,2022-04-13,2022-04-26,2022-05-16,13,20,33,red prestacional sabogal,cusco,legal entity,cefalexina 500 mg,adjudicacion simplificada,0,0,0,101165.11,2022,0
1,2023-12-15,2024-02-03,2024-03-19,50,45,95,red prestacional lambayeque,lima,legal entity,hipromelosa 0.3%,licitacion publica,0,0,1,694535.50,2023,1
2,2022-09-28,2022-10-28,2022-12-26,30,59,89,red prestacional almenara,lambayeque,legal entity,meropenem 500 mg,licitacion publica,0,1,1,58864.57,2022,0
3,2025-04-17,2025-05-26,2025-06-28,39,33,72,red prestacional rebagliati,madre de dios,consortium,ceftriaxona 1 g,licitacion publica,1,1,1,24399.68,2025,0
4,2023-03-13,2023-04-07,2023-04-18,25,11,36,red asistencial madre de dios,lima,legal entity,ceftriaxona 1 g,adjudicacion simplificada,0,0,1,197551.31,2023,0


In [2]:
# Diagnóstico estructural: tipos de dato y nulidad
diagnostic = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_nulos': df.isnull().sum(),
    'pct_nulos': (df.isnull().mean() * 100).round(2)
})
diagnostic

,dtype,n_nulos,pct_nulos
fecha_convocatoria,str,0,0.0
fecha_buena_pro,str,0,0.0
fecha_suscripcion,str,0,0.0
lead_time_adjudicacion,int64,0,0.0
lead_time_formalizacion,int64,0,0.0
lead_time_total_proceso,int64,0,0.0
red_asistencial,str,0,0.0
dpto_entrega_item,str,0,0.0
tipo_proveedor,str,0,0.0
denominacion_dci,str,0,0.0


## 3. Conversión Segura de Tipos Temporales

Las columnas `fecha_convocatoria`, `fecha_buena_pro` y `fecha_suscripcion` se encuentran serializadas como cadenas de texto (`YYYY-MM-DD`). Se ejecuta la conversión con `pd.to_datetime` empleando `errors='coerce'`, que transforma cualquier registro con formato corrupto o no parseable en `NaT` (Not a Time), evitando la propagación de excepciones y permitiendo un tratamiento posterior explícito de la nulidad temporal inducida.


In [3]:
date_cols = ['fecha_convocatoria', 'fecha_buena_pro', 'fecha_suscripcion']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')

# Verificación de fechas no parseables (NaT inducidos por 'coerce')
nat_report = df[date_cols].isnull().sum()
print("Registros con fecha no parseable (NaT) por columna:")
print(nat_report)

# Eliminación justificada: un registro con fecha ancla inválida no permite
# calcular ningún Lead Time consistente, por lo que se descarta.
n_before = len(df)
df = df.dropna(subset=date_cols).reset_index(drop=True)
print(f"\nRegistros descartados por fecha no parseable: {n_before - len(df)}")
print(f"Registros remanentes: {len(df)}")

Registros con fecha no parseable (NaT) por columna:
fecha_convocatoria    0
fecha_buena_pro       0
fecha_suscripcion     0
dtype: int64

Registros descartados por fecha no parseable: 0
Registros remanentes: 12500


## 4. Cálculo e Ingeniería de las Variables de Lead Time

Se recalculan programáticamente los tres tiempos de ciclo a partir de las fechas normalizadas, expresados en días enteros (`.dt.days`). Aunque el dataset de origen ya contiene columnas homónimas, se realiza el recálculo explícito como control de calidad (Data Quality Gate) para garantizar consistencia matemática entre las fechas crudas y los tiempos de ciclo derivados, evitando arrastrar errores de un preprocesamiento previo no auditado.


In [4]:
df['lead_time_adjudicacion_calc'] = (df['fecha_buena_pro'] - df['fecha_convocatoria']).dt.days
df['lead_time_formalizacion_calc'] = (df['fecha_suscripcion'] - df['fecha_buena_pro']).dt.days
df['lead_time_total_proceso_calc'] = (df['fecha_suscripcion'] - df['fecha_convocatoria']).dt.days

# Auditoría de consistencia contra las columnas originales del staging
for base, calc in [('lead_time_adjudicacion', 'lead_time_adjudicacion_calc'),
                    ('lead_time_formalizacion', 'lead_time_formalizacion_calc'),
                    ('lead_time_total_proceso', 'lead_time_total_proceso_calc')]:
    mismatches = (df[base] != df[calc]).sum()
    print(f"{base}: {mismatches} discrepancias frente al recálculo desde fechas crudas")

# Se adoptan los Lead Times recalculados como fuente única de verdad (single source of truth)
df['lead_time_adjudicacion'] = df['lead_time_adjudicacion_calc']
df['lead_time_formalizacion'] = df['lead_time_formalizacion_calc']
df['lead_time_total_proceso'] = df['lead_time_total_proceso_calc']
df.drop(columns=['lead_time_adjudicacion_calc', 'lead_time_formalizacion_calc',
                  'lead_time_total_proceso_calc'], inplace=True)

df[['lead_time_adjudicacion', 'lead_time_formalizacion', 'lead_time_total_proceso']].describe()

lead_time_adjudicacion: 0 discrepancias frente al recálculo desde fechas crudas
lead_time_formalizacion: 0 discrepancias frente al recálculo desde fechas crudas
lead_time_total_proceso: 0 discrepancias frente al recálculo desde fechas crudas


,lead_time_adjudicacion,lead_time_formalizacion,lead_time_total_proceso
count,12500.000000,12500.000000,12500.000000
mean,32.771760,22.697600,55.469360
std,19.413527,12.632506,28.933903
min,1.000000,1.000000,3.000000
25%,18.000000,13.000000,33.000000
50%,28.000000,20.000000,48.000000
75%,45.000000,30.000000,75.000000
max,120.000000,83.000000,169.000000


## 5. Tratamiento de Inconsistencias Lógicas (Fechas Invertidas)

Una inconsistencia lógica ocurre cuando la fecha de un hito posterior antecede a la de un hito anterior en el flujo administrativo (p. ej., `fecha_buena_pro < fecha_convocatoria`), lo cual es administrativamente imposible y produce Lead Times negativos. Estos casos se tratan como **anomalías de captura de datos** y se corrigen mediante **imputación por la mediana** de la variable correspondiente, preservando el registro (dado que las demás 13 variables predictoras siguen siendo válidas) en lugar de eliminarlo, salvo que la proporción de anomalías sea marginal, en cuyo caso la eliminación es igualmente defendible. Se documenta el criterio y el conteo de forma explícita.


In [5]:
leadtime_cols = ['lead_time_adjudicacion', 'lead_time_formalizacion', 'lead_time_total_proceso']

anomaly_summary = {}
for col in leadtime_cols:
    n_negative = (df[col] < 0).sum()
    anomaly_summary[col] = n_negative

print("Conteo de Lead Times negativos (inconsistencia lógica) por variable:")
for k, v in anomaly_summary.items():
    print(f"  {k}: {v} ({v/len(df)*100:.3f}%)")

# Criterio de tratamiento: imputación por mediana condicional (robusta a asimetría)
for col in leadtime_cols:
    mask_neg = df[col] < 0
    if mask_neg.sum() > 0:
        median_val = df.loc[~mask_neg, col].median()
        df.loc[mask_neg, col] = median_val
        print(f"Imputados {mask_neg.sum()} registros en '{col}' con mediana = {median_val}")

print("\nValidación post-corrección — mínimos por columna:")
print(df[leadtime_cols].min())

Conteo de Lead Times negativos (inconsistencia lógica) por variable:
  lead_time_adjudicacion: 0 (0.000%)
  lead_time_formalizacion: 0 (0.000%)
  lead_time_total_proceso: 0 (0.000%)

Validación post-corrección — mínimos por columna:
lead_time_adjudicacion     1
lead_time_formalizacion    1
lead_time_total_proceso    3
dtype: int64


## 6. Tratamiento de Outliers mediante IQR

Se aplica el método del **Rango Intercuartílico (IQR)** sobre cada variable de Lead Time para mitigar la influencia de procesos de adquisición atípicamente paralizados (colas superiores extensas, propias de licitaciones impugnadas o desabastecimiento crítico) o irrealmente acelerados. Los límites de corte se definen como:

$$LI = Q_1 - 1.5 \cdot IQR \qquad LS = Q_3 + 1.5 \cdot IQR$$

En lugar de eliminar los registros extremos —lo cual reduciría el tamaño muestral y podría eliminar precisamente los casos de mayor riesgo—, se aplica **capping (winsorización)**, acotando los valores fuera de rango a los límites calculados. Esta decisión preserva la señal predictiva de "proceso anómalamente lento" sin distorsionar la escala de la variable.


In [6]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_report = []
for col in leadtime_cols:
    li, ls = iqr_bounds(df[col])
    n_out = ((df[col] < li) | (df[col] > ls)).sum()
    outlier_report.append({'variable': col, 'limite_inferior': round(li, 2),
                            'limite_superior': round(ls, 2), 'n_outliers': n_out,
                            'pct_outliers': round(n_out/len(df)*100, 2)})
    df[col] = df[col].clip(lower=max(li, 0), upper=ls)

pd.DataFrame(outlier_report)

,variable,limite_inferior,limite_superior,n_outliers,pct_outliers
0,lead_time_adjudicacion,-22.5,85.5,115,0.92
1,lead_time_formalizacion,-12.5,55.5,175,1.40
2,lead_time_total_proceso,-30.0,138.0,70,0.56


## 7. Codificación de Variables Categóricas

XGBoost y Random Forest requieren representaciones puramente numéricas. Se adopta una estrategia dual según la cardinalidad de cada variable categórica:

- **One-Hot Encoding** (con `drop_first=True` para evitar la trampa de la multicolinealidad / *dummy variable trap*) para variables de **baja cardinalidad**: `red_asistencial`, `dpto_entrega_item`, `tipo_proveedor`, `tipo_proceso_seleccion`.
- **Target Encoding** (media suavizada del target por categoría, vía `category_encoders.TargetEncoder`) para `denominacion_dci`, la variable de **mayor cardinalidad relativa**, evitando la explosión dimensional que produciría un One-Hot sobre decenas de principios activos distintos, y capturando directamente la tasa histórica de adendas asociada a cada fármaco.

El Target Encoder se ajusta **exclusivamente sobre la partición de entrenamiento** en el Notebook 2 para prevenir fuga de información (*target leakage*); en este módulo se deja preparado el pipeline y se genera una versión de referencia ajustada sobre el dataset completo únicamente para fines de exportación y análisis exploratorio, dejando explícito que el ajuste real de producción ocurre post-split.


In [7]:
import category_encoders as ce

low_card_cols = ['red_asistencial', 'dpto_entrega_item', 'tipo_proveedor', 'tipo_proceso_seleccion']
high_card_col = 'denominacion_dci'

print("Cardinalidad de variables categóricas:")
for c in low_card_cols + [high_card_col]:
    print(f"  {c}: {df[c].nunique()} categorías")

# One-Hot Encoding (baja cardinalidad)
df_ohe = pd.get_dummies(df, columns=low_card_cols, drop_first=True, prefix=low_card_cols)

# Target Encoding de referencia (ajuste completo, solo para exportación/EDA;
# el ajuste válido para modelado se realiza sobre el train set en el Notebook 2)
target_encoder_ref = ce.TargetEncoder(cols=[high_card_col], smoothing=0.3)
df_ohe[f'{high_card_col}_target_enc'] = target_encoder_ref.fit_transform(
    df_ohe[high_card_col], df_ohe['flag_tiene_adenda']
)

df_ohe.head()

Cardinalidad de variables categóricas:
  red_asistencial: 9 categorías
  dpto_entrega_item: 10 categorías
  tipo_proveedor: 3 categorías
  tipo_proceso_seleccion: 4 categorías
  denominacion_dci: 15 categorías


,fecha_convocatoria,fecha_buena_pro,fecha_suscripcion,lead_time_adjudicacion,lead_time_formalizacion,lead_time_total_proceso,denominacion_dci,es_consorcio,es_uso_critico,es_uso_intrahospitalario,monto_adjudicado_soles,anio,flag_tiene_adenda,red_asistencial_red asistencial arequipa,red_asistencial_red asistencial la libertad,red_asistencial_red asistencial loreto,red_asistencial_red asistencial madre de dios,red_asistencial_red prestacional almenara,red_asistencial_red prestacional lambayeque,red_asistencial_red prestacional rebagliati,red_asistencial_red prestacional sabogal,dpto_entrega_item_apurimac,dpto_entrega_item_arequipa,dpto_entrega_item_cusco,dpto_entrega_item_la libertad,dpto_entrega_item_lambayeque,dpto_entrega_item_lima,dpto_entrega_item_loreto,dpto_entrega_item_madre de dios,dpto_entrega_item_piura,tipo_proveedor_legal entity,tipo_proveedor_natural person,tipo_proceso_seleccion_contratacion directa,tipo_proceso_seleccion_licitacion publica,tipo_proceso_seleccion_subasta inversa electronica,denominacion_dci_target_enc
0,2022-04-13,2022-04-26,2022-05-16,13.0,20.0,33,cefalexina 500 mg,0,0,0,101165.11,2022,0,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,0.161465
1,2023-12-15,2024-02-03,2024-03-19,50.0,45.0,95,hipromelosa 0.3%,0,0,1,694535.50,2023,1,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,True,False,False,True,False,0.180543
2,2022-09-28,2022-10-28,2022-12-26,30.0,55.5,89,meropenem 500 mg,0,1,1,58864.57,2022,0,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True,False,0.305069
3,2025-04-17,2025-05-26,2025-06-28,39.0,33.0,72,ceftriaxona 1 g,1,1,1,24399.68,2025,0,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,0.282110
4,2023-03-13,2023-04-07,2023-04-18,25.0,11.0,36,ceftriaxona 1 g,0,0,1,197551.31,2023,0,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,0.282110


## 8. Escalamiento de Variables Numéricas

Se aplica `RobustScaler` sobre las variables numéricas continuas (`monto_adjudicado_soles` y los tres Lead Times), dado que este escalador utiliza la mediana y el rango intercuartílico como estadísticos de centrado y dispersión, siendo menos sensible a la cola residual de outliers no eliminados por el capping IQR que `StandardScaler` (basado en media y desviación estándar). Se conserva una copia de las variables originales sin escalar (`_raw`) para fines de interpretabilidad y visualización en el Notebook 2.


In [8]:
from sklearn.preprocessing import RobustScaler

numeric_to_scale = ['lead_time_adjudicacion', 'lead_time_formalizacion',
                     'lead_time_total_proceso', 'monto_adjudicado_soles']

# Preservar versión cruda para interpretabilidad / EDA posterior
for col in numeric_to_scale:
    df_ohe[f'{col}_raw'] = df_ohe[col]

scaler = RobustScaler()
df_ohe[numeric_to_scale] = scaler.fit_transform(df_ohe[numeric_to_scale])

df_ohe[[c for c in df_ohe.columns if 'lead_time' in c or 'monto' in c]].describe()

,lead_time_adjudicacion,lead_time_formalizacion,lead_time_total_proceso,monto_adjudicado_soles,lead_time_adjudicacion_raw,lead_time_formalizacion_raw,lead_time_total_proceso_raw,monto_adjudicado_soles_raw
count,12500.000000,12500.000000,12500.000000,12500.000000,12500.00000,12500.00000,12500.000000,1.250000e+04
mean,0.174096,0.154482,0.176623,0.710743,32.70060,22.62620,55.418160,7.649251e+05
std,0.710892,0.730411,0.685094,2.579178,19.19408,12.41698,28.773945,1.673841e+06
min,-1.000000,-1.117647,-1.071429,-0.466792,1.00000,1.00000,3.000000,7.254300e+02
25%,-0.370370,-0.411765,-0.357143,-0.281521,18.00000,13.00000,33.000000,1.209629e+05
50%,0.000000,0.000000,0.000000,0.000000,28.00000,20.00000,48.000000,3.036652e+05
75%,0.629630,0.588235,0.642857,0.718479,45.00000,30.00000,75.000000,7.699451e+05
max,2.129630,2.088235,2.142857,61.703121,85.50000,55.50000,138.000000,4.034789e+07


## 9. Exportación de la Matriz Final

Se descartan las columnas de fecha cruda (ya explotadas en Lead Times) y la variable categórica textual `denominacion_dci` (sustituida por su codificación numérica), conservando el resto de variables predictoras, los Lead Times escalados y crudos, y la variable objetivo. La matriz resultante se exporta a `data/medicine_overrun_dataset.csv` para ser consumida íntegramente por el Notebook 2 de entrenamiento y evaluación.


In [9]:
cols_to_drop = date_cols + [high_card_col]
final_df = df_ohe.drop(columns=cols_to_drop)

# Reordenar: identificadores/predictores primero, target al final
target_col = 'flag_tiene_adenda'
ordered_cols = [c for c in final_df.columns if c != target_col] + [target_col]
final_df = final_df[ordered_cols]

OUTPUT_PATH = '../data/medicine_overrun_dataset.csv'
final_df.to_csv(OUTPUT_PATH, index=False)

print(f"Dataset final exportado: {OUTPUT_PATH}")
print(f"Dimensiones finales: {final_df.shape[0]} registros x {final_df.shape[1]} columnas")
print(f"\nBalance de la variable objetivo 'flag_tiene_adenda':")
print(final_df[target_col].value_counts(normalize=True).round(4))
final_df.head()

Dataset final exportado: ../data/medicine_overrun_dataset.csv
Dimensiones finales: 12500 registros x 36 columnas

Balance de la variable objetivo 'flag_tiene_adenda':
flag_tiene_adenda
0    0.7717
1    0.2283
Name: proportion, dtype: float64


,lead_time_adjudicacion,lead_time_formalizacion,lead_time_total_proceso,es_consorcio,es_uso_critico,es_uso_intrahospitalario,monto_adjudicado_soles,anio,red_asistencial_red asistencial arequipa,red_asistencial_red asistencial la libertad,red_asistencial_red asistencial loreto,red_asistencial_red asistencial madre de dios,red_asistencial_red prestacional almenara,red_asistencial_red prestacional lambayeque,red_asistencial_red prestacional rebagliati,red_asistencial_red prestacional sabogal,dpto_entrega_item_apurimac,dpto_entrega_item_arequipa,dpto_entrega_item_cusco,dpto_entrega_item_la libertad,dpto_entrega_item_lambayeque,dpto_entrega_item_lima,dpto_entrega_item_loreto,dpto_entrega_item_madre de dios,dpto_entrega_item_piura,tipo_proveedor_legal entity,tipo_proveedor_natural person,tipo_proceso_seleccion_contratacion directa,tipo_proceso_seleccion_licitacion publica,tipo_proceso_seleccion_subasta inversa electronica,denominacion_dci_target_enc,lead_time_adjudicacion_raw,lead_time_formalizacion_raw,lead_time_total_proceso_raw,monto_adjudicado_soles_raw,flag_tiene_adenda
0,-0.555556,0.000000,-0.357143,0,0,0,-0.312027,2022,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,0.161465,13.0,20.0,33,101165.11,0
1,0.814815,1.470588,1.119048,0,0,1,0.602282,2023,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,True,False,False,True,False,0.180543,50.0,45.0,95,694535.50,1
2,0.074074,2.088235,0.976190,0,1,1,-0.377207,2022,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True,False,0.305069,30.0,55.5,89,58864.57,0
3,0.407407,0.764706,0.571429,1,1,1,-0.430313,2025,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,0.282110,39.0,33.0,72,24399.68,0
4,-0.111111,-0.529412,-0.285714,0,0,1,-0.163508,2023,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,0.282110,25.0,11.0,36,197551.31,0
